# Gradient × update token MLPs

This notebook trains only the two models needed for the U–G comparison:

- **MLP-U:** trained on understanding gradient × update scores.
- **MLP-G:** trained on generation gradient × update scores.

It uses two training seeds, splits complete examples into train/validation/test, and keeps all valid token types. The target is `log(score + 1e-12)`, because gradient scores can be very small. Generation labels use the self-generated reference images saved by the collection script, so they are pseudo-target flow losses rather than ground-truth training losses.

Run the five code cells in order. Compressed generation files are decompressed once into Colab's local disk cache.

In [ ]:
# 1. Paths, settings, examples, and train/validation/test split
import gc, gzip, hashlib, json, re, shutil
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/intership")
OUTPUT_DIR = DATA_ROOT / "gradient_mlp_results" / "two_seed_run"
CACHE_DIR = Path("/content/gradient_mlp_cache")

U_NAMES = ["counting", "spatial_relations", "attribute_combinations", "visual_state_traffic_lights"]
FOLDERS = {f"U/{name}": DATA_ROOT / "gradient_update_understanding" / name for name in U_NAMES}
FOLDERS["G/counting_spatial"] = DATA_ROOT / "gradient_update_generation" / "counting_spatial"

TRAIN_SEEDS = [42, 43]
SPLIT_SEED = 42
TOP_K_FRACTION = 0.20
HIDDEN_WIDTH = 128
BATCH_SIZE = 256
MAX_EPOCHS, UPDATES_PER_EPOCH, PATIENCE = 15, 150, 3
VAL_BATCHES = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

examples = []
for category, folder in FOLDERS.items():
    task = category[0]
    expected = 6 if task == "U" else 12
    paths = sorted(p for p in folder.glob("sample_*.pt") if re.fullmatch(r"sample_\d+\.pt", p.name))
    if len(paths) != expected:
        raise ValueError(f"{folder}: expected {expected} samples, found {len(paths)}")

    for aggregate in paths:
        data = torch.load(aggregate, map_location="cpu", weights_only=True, mmap=True)
        layers = list(map(int, data["layer_indices"]))
        if task == "U":
            if data["score_name"] != "token_gradient_update_answer_loss_v1":
                raise ValueError(f"Wrong U score file: {aggregate}")
            files, steps = [aggregate], [-1]
        else:
            if data["score_name"] != "token_gradient_update_flow_loss_v1":
                raise ValueError(f"Wrong G score file: {aggregate}")
            steps = list(map(int, data["step_indices"]))
            index = int(data["sample"]["record_index"])
            files = [folder / f"sample_{index:06d}_step_{step:04d}_conditional.pt.gz" for step in steps]
            if not all(path.is_file() for path in files):
                raise FileNotFoundError(f"Missing G step files for {aggregate}")
        examples.append(dict(
            uid=f"{category}/{aggregate.stem}", task=task, category=category,
            aggregate=aggregate, files=files, steps=steps, layers=layers,
        ))
        del data

layer_sets = {tuple(example["layers"]) for example in examples}
if len(layer_sets) != 1:
    raise ValueError("All runs must contain the same layers")
LAYER_IDS = list(next(iter(layer_sets)))

rng = np.random.default_rng(SPLIT_SEED)
for category in [f"U/{name}" for name in U_NAMES]:
    group = [example for example in examples if example["category"] == category]
    order = rng.permutation(len(group))
    for rank, index in enumerate(order):
        group[index]["split"] = "train" if rank < 4 else "val" if rank == 4 else "test"

group = [example for example in examples if example["task"] == "G"]
order = rng.permutation(len(group))
for rank, index in enumerate(order):
    group[index]["split"] = "train" if rank < 8 else "val" if rank < 10 else "test"

first_u = next(example for example in examples if example["task"] == "U")
first_data = torch.load(first_u["aggregate"], map_location="cpu", weights_only=True, mmap=True)
FEATURE_DIM = int(first_data["layers"][str(LAYER_IDS[0])]["input_features"].shape[1])
del first_data

display(pd.DataFrame(examples).groupby(["task", "split"]).size().unstack(fill_value=0))
print("Device:", DEVICE, "| Feature dimension:", FEATURE_DIM, "| Results:", OUTPUT_DIR)

In [ ]:
# 2. Lightweight file cache, batches, and MLP
def local_file(source):
    source = Path(source)
    if source.suffix != ".gz":
        return source
    stat = source.stat()
    key = hashlib.sha1(f"{source}:{stat.st_size}:{stat.st_mtime_ns}".encode()).hexdigest()
    destination = CACHE_DIR / f"{key}.pt"
    if not destination.exists():
        temporary = destination.with_suffix(".tmp")
        with gzip.open(source, "rb") as src, temporary.open("wb") as dst:
            shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
        temporary.replace(destination)
    return destination

@lru_cache(maxsize=4)
def load_payload(source):
    return torch.load(local_file(source), map_location="cpu", weights_only=True, mmap=True)

train_pool = {task: [e for e in examples if e["task"] == task and e["split"] == "train"] for task in ("U", "G")}
val_pool = {task: [e for e in examples if e["task"] == task and e["split"] == "val"] for task in ("U", "G")}
test_pool = [e for e in examples if e["split"] == "test"]

def sample_batch(pool, rng):
    xs, layer_codes, ys = [], [], []
    while sum(len(y) for y in ys) < BATCH_SIZE:
        example = pool[int(rng.integers(len(pool)))]
        source = example["files"][int(rng.integers(len(example["files"])))]
        data = load_payload(str(source))
        layer_code = int(rng.integers(len(LAYER_IDS)))
        values = data["layers"][str(LAYER_IDS[layer_code])]
        valid = torch.arange(len(data["tokens"]))
        if example["task"] == "G":
            valid = data["valid_token_mask"].nonzero().flatten()
        count = min(64, BATCH_SIZE - sum(len(y) for y in ys))
        positions = valid[torch.from_numpy(rng.integers(len(valid), size=count))]
        xs.append(values["input_features"].index_select(0, positions).float())
        ys.append(values["gradient_update"].index_select(0, positions).float())
        layer_codes.append(torch.full((count,), layer_code, dtype=torch.long))
    return torch.cat(xs), torch.cat(layer_codes), torch.cat(ys)

class TokenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(FEATURE_DIM + 1 + len(LAYER_IDS), HIDDEN_WIDTH),
            nn.GELU(),
            nn.Linear(HIDDEN_WIDTH, 1),
        )

    def forward(self, hidden, layer):
        rms = hidden.square().mean(1, keepdim=True).sqrt().clamp_min(1e-6)
        layer_one_hot = nn.functional.one_hot(layer, len(LAYER_IDS)).float()
        features = torch.cat([hidden / rms, torch.log(rms), layer_one_hot], dim=1)
        return self.net(features).squeeze(1)

def target(score):
    return torch.log(score.clamp_min(1e-12))

validation = {}
for task in ("U", "G"):
    val_rng = np.random.default_rng(SPLIT_SEED + (0 if task == "U" else 1000))
    validation[task] = [sample_batch(val_pool[task], val_rng) for _ in range(VAL_BATCHES)]

print("Prepared fixed validation batches")

In [ ]:
# 3. Train MLP-U and MLP-G with two seeds
@torch.inference_mode()
def validation_loss(model, batches):
    model.eval()
    losses = []
    for hidden, layer, score in batches:
        prediction = model(hidden.to(DEVICE), layer.to(DEVICE))
        losses.append(nn.functional.mse_loss(prediction, target(score.to(DEVICE))).item())
    return float(np.mean(losses))

history = []
for model_name, task in [("MLP-U", "U"), ("MLP-G", "G")]:
    for seed in TRAIN_SEEDS:
        checkpoint = OUTPUT_DIR / f"{model_name}_seed_{seed}.pt"
        if checkpoint.exists():
            print("Reusing:", checkpoint)
            continue

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        rng = np.random.default_rng(seed + (0 if task == "U" else 1000))
        model = TokenMLP().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        best_loss, stale, best_state = float("inf"), 0, None

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            train_losses = []
            for _ in range(UPDATES_PER_EPOCH):
                hidden, layer, score = sample_batch(train_pool[task], rng)
                loss = nn.functional.mse_loss(
                    model(hidden.to(DEVICE), layer.to(DEVICE)),
                    target(score.to(DEVICE)),
                )
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                train_losses.append(loss.item())

            val = validation_loss(model, validation[task])
            history.append(dict(model=model_name, seed=seed, epoch=epoch,
                                train_loss=np.mean(train_losses), val_loss=val))
            print(f"{model_name} seed={seed} epoch={epoch}: val={val:.4f}")
            if val < best_loss:
                best_loss, stale = val, 0
                best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            else:
                stale += 1
            if stale >= PATIENCE:
                break

        torch.save({"state_dict": best_state, "model": model_name, "seed": seed,
                    "layers": LAYER_IDS, "feature_dim": FEATURE_DIM}, checkpoint)
        print("Saved:", checkpoint)

if history:
    pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

In [ ]:
# 4. Test accuracy and direct MLP-U versus MLP-G overlap
models = {}
for name in ("MLP-U", "MLP-G"):
    for seed in TRAIN_SEEDS:
        checkpoint = torch.load(OUTPUT_DIR / f"{name}_seed_{seed}.pt", map_location="cpu", weights_only=True)
        model = TokenMLP().to(DEVICE)
        model.load_state_dict(checkpoint["state_dict"])
        model.eval()
        models[name, seed] = model

@torch.inference_mode()
def predict(model, hidden, layer_code):
    layer = torch.full((len(hidden),), layer_code, dtype=torch.long, device=DEVICE)
    return model(hidden.float().to(DEVICE), layer).cpu().numpy()

def selected(values, k):
    return np.argsort(-np.asarray(values), kind="stable")[:k]

rows = []
for example in test_pool:
    for file_index, source in enumerate(example["files"]):
        data = load_payload(str(source))
        valid = torch.arange(len(data["tokens"]))
        if example["task"] == "G":
            valid = data["valid_token_mask"].nonzero().flatten()
        step = example["steps"][file_index]

        for layer_code, layer in enumerate(LAYER_IDS):
            values = data["layers"][str(layer)]
            hidden = values["input_features"].index_select(0, valid)
            truth = values["gradient_update"].index_select(0, valid).float().numpy()
            k = max(1, int(np.ceil(TOP_K_FRACTION * len(valid))))
            truth_top = selected(truth, k)

            for seed in TRAIN_SEEDS:
                prediction_u = predict(models["MLP-U", seed], hidden, layer_code)
                prediction_g = predict(models["MLP-G", seed], hidden, layer_code)
                top_u, top_g = selected(prediction_u, k), selected(prediction_g, k)
                rows.append(dict(
                    task=example["task"], uid=example["uid"], seed=seed,
                    layer=layer, step=step, chance=k / len(valid),
                    overlap=np.intersect1d(top_u, top_g).size / k,
                    mlp_u_target=np.intersect1d(top_u, truth_top).size / k,
                    mlp_g_target=np.intersect1d(top_g, truth_top).size / k,
                ))
        gc.collect()

details = pd.DataFrame(rows)
EVAL_DIR = OUTPUT_DIR / "evaluation_top20pct"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
details.to_csv(EVAL_DIR / "overlap_details.csv", index=False)

by_layer_step = (
    details.groupby(["task", "layer", "step"], as_index=False)
    .agg(overlap=("overlap", "mean"), chance=("chance", "mean"))
)
by_layer_step.to_csv(EVAL_DIR / "overlap_by_layer_step.csv", index=False)

quality = pd.DataFrame({
    task: [
        details.loc[details.task == task, "mlp_u_target"].mean(),
        details.loc[details.task == task, "mlp_g_target"].mean(),
        details.loc[details.task == task, "chance"].mean(),
    ] for task in ("U", "G")
}, index=["MLP-U", "MLP-G", "Random expectation"])
quality.columns = ["On U test", "On G test"]
quality.to_csv(EVAL_DIR / "predictor_topk_accuracy.csv")
display(quality.style.format("{:.1%}"))

In [ ]:
# 5. Final U–G overlap graphs
# U inputs: overlap by layer
u_plot = (
    by_layer_step[by_layer_step["task"] == "U"]
    .groupby("layer", as_index=False)
    .agg(overlap=("overlap", "mean"), chance=("chance", "mean"))
)
chance_u = u_plot["chance"].mean()

plt.figure(figsize=(9, 5))
plt.plot(u_plot["layer"], u_plot["overlap"], marker="o", linewidth=2)
plt.axhline(chance_u, color="red", linestyle="--", label=f"Random ≈ {chance_u:.1%}")
plt.title("MLP-U vs MLP-G on U test samples — all tokens")
plt.xlabel("Layer"); plt.ylabel("Top-K overlap"); plt.ylim(0, 1)
plt.grid(alpha=0.2); plt.legend(); plt.tight_layout()
plt.savefig(EVAL_DIR / "direct_overlap_U_all_tokens.png", dpi=200, bbox_inches="tight")
plt.show()

# G inputs: overlap by layer and generation step
g_data = by_layer_step[by_layer_step["task"] == "G"].copy()
g_map = g_data.pivot(index="step", columns="layer", values="overlap")
chance_g = g_data["chance"].mean()
vmax = max(0.60, chance_g + 0.01, float(np.nanmax(g_map.values)) + 0.01)

fig, ax = plt.subplots(figsize=(14, 7))
image = ax.imshow(
    g_map.values, aspect="auto", origin="upper", cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=0, vcenter=chance_g, vmax=vmax),
)
x_ticks = np.arange(0, len(g_map.columns), 2)
y_ticks = np.arange(0, len(g_map.index), 5)
ax.set_xticks(x_ticks, [g_map.columns[i] for i in x_ticks])
ax.set_yticks(y_ticks, [g_map.index[i] for i in y_ticks])
ax.set_title("MLP-U vs MLP-G on G test samples — all tokens")
ax.set_xlabel("Layer"); ax.set_ylabel("Generation step")
fig.colorbar(image, ax=ax).set_label("Top-K overlap")
plt.tight_layout()
plt.savefig(EVAL_DIR / "direct_overlap_G_all_tokens.png", dpi=200, bbox_inches="tight")
plt.show()

# G inputs: all generation steps averaged
g_plot = (
    g_data.groupby("layer", as_index=False)
    .agg(overlap=("overlap", "mean"), chance=("chance", "mean"))
)
plt.figure(figsize=(9, 5))
plt.plot(g_plot["layer"], g_plot["overlap"], marker="o", linewidth=2)
plt.axhline(g_plot["chance"].mean(), color="red", linestyle="--",
            label=f"Random ≈ {g_plot['chance'].mean():.1%}")
plt.title("MLP-U vs MLP-G on G test samples — all steps averaged")
plt.xlabel("Layer"); plt.ylabel("Top-K overlap"); plt.ylim(0, 1)
plt.grid(alpha=0.2); plt.legend(); plt.tight_layout()
plt.savefig(EVAL_DIR / "direct_overlap_G_by_layer.png", dpi=200, bbox_inches="tight")
plt.show()

# G inputs: all layers averaged at each generation step
g_by_step = (
    g_data.groupby("step", as_index=False)
    .agg(overlap=("overlap", "mean"), chance=("chance", "mean"))
)
plt.figure(figsize=(9, 5))
plt.plot(g_by_step["step"], g_by_step["overlap"], marker="o")
plt.axhline(g_by_step["chance"].mean(), color="red", linestyle="--",
            label=f"Random ≈ {g_by_step['chance'].mean():.1%}")
plt.title("U–G overlap across generation steps — all tokens")
plt.xlabel("Generation step"); plt.ylabel("Top-K overlap"); plt.ylim(0, 1)
plt.grid(alpha=0.2); plt.legend(); plt.tight_layout()
plt.savefig(EVAL_DIR / "direct_overlap_G_by_step.png", dpi=200, bbox_inches="tight")
plt.show()

print("Results saved to:", EVAL_DIR)